In [0]:
USE dinedash;

In [0]:
SELECT
    o.customer_id,
    c.name AS customer_name,
    r.restaurant_name,
    COUNT(*) AS total_orders
FROM orders_bronze o
JOIN customers_filtered c
    ON o.customer_id = c.customer_id
JOIN restaurants_filtered r
    ON o.restaurant_id = r.restaurant_id
GROUP BY
    o.customer_id,
    customer_name,
    restaurant_name
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY o.customer_id
    ORDER BY total_orders DESC
) = 1;

In [0]:
SELECT
    DATE_FORMAT(o.timestamp,'yyyy-MM') AS order_month,
    l.area,
    COUNT(*) AS total_orders
FROM orders_bronze o
JOIN locations_filtered l
ON o.delivery_location_id=l.location_id
GROUP BY
    order_month,
    l.area
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY order_month
    ORDER BY total_orders DESC
)<=10;

In [0]:
SELECT
    o.order_id,
    c.name AS customer_name,
    r.restaurant_name,
    o.total_amount,
    o.payment_method,
    o.order_status
FROM orders_bronze o
JOIN customers_filtered c
    ON o.customer_id = c.customer_id
JOIN restaurants_filtered r
    ON o.restaurant_id = r.restaurant_id
WHERE LOWER(o.order_status) = 'delivered'
  AND LOWER(o.payment_method) = 'card';

In [0]:
SELECT
    r.cuisines,
    ROUND(AVG(o.total_amount),2) AS avg_order_value
FROM orders_bronze o
JOIN restaurants_filtered r
USING (restaurant_id)
GROUP BY r.cuisines
ORDER BY avg_order_value DESC;

In [0]:
SELECT
    d.agent_id,
    d.name,
    COUNT(*) AS high_value_orders
FROM orders_bronze o  
JOIN delivery_agents_filtered d 
USING(agent_id)
WHERE o.total_amount>50
GROUP BY d.agent_id,d.name
ORDER BY high_value_orders DESC;

In [0]:
SELECT
    l.area,
    ROUND(SUM(o.total_amount),2) as revenue
FROM orders_bronze o
JOIN locations_filtered l 
ON o.delivery_location_id=l.location_id
JOIN restaurants_filtered r
USING(restaurant_id)
WHERE LOWER(r.cuisines)='italian'
GROUP BY l.area
ORDER BY revenue DESC;

In [0]:
SELECT
    r.cuisines,
    ROUND(AVG(try_divide(o.tip, o.total_amount)), 2) AS avg_tip_ratio
FROM orders_bronze o  
JOIN restaurants_filtered r 
USING(restaurant_id)
GROUP BY r.cuisines
ORDER BY avg_tip_ratio DESC;

In [0]:
WITH exploded_orders AS (
    SELECT
        o.restaurant_id,
        EXPLODE(o.items_ordered) AS item
    FROM orders_bronze o
)

SELECT
    m.item_name,
    r.restaurant_name,
    COUNT(*) AS total_orders
FROM exploded_orders eo
JOIN menu_items_filtered m
    ON eo.item.item_id = m.item_id
JOIN restaurants_filtered r
    ON eo.restaurant_id = r.restaurant_id
GROUP BY
    m.item_name,
    r.restaurant_name
ORDER BY total_orders DESC;

In [0]:
SELECT
    a.agent_id,
    a.name,
    COUNT(DISTINCT r.cuisines) as diverse_cuisines
FROM orders_bronze o
JOIN delivery_agents_filtered a 
USING (agent_id)
JOIN restaurants_filtered r 
USING (restaurant_id)
GROUP BY a.agent_id, a.name
ORDER BY diverse_cuisines DESC;

In [0]:
WITH customer_spend AS( 
    SELECT
        o.customer_id,
        EXTRACT(YEAR FROM c.signup_date) AS signup_year,
        SUM(o.total_amount) AS lifetime_value
    FROM customers_filtered c  
    JOIN orders_bronze o 
    USING (customer_id)
    GROUP BY signup_year,o.customer_id
)
SELECT 
    signup_year,
    ROUND(AVG(lifetime_value),2) AS avg_cust_ltv
FROM customer_spend
GROUP BY signup_year
ORDER BY signup_year

In [0]:
SELECT
    l.location_id,
    l.area,
    r.restaurant_name,
    COUNT(*) AS total_orders
FROM orders_bronze o
JOIN restaurants_filtered r
    USING (restaurant_id)
JOIN locations_filtered l
    ON o.delivery_location_id = l.location_id
GROUP BY
    l.location_id,
    l.area,
    r.restaurant_name
QUALIFY RANK() OVER (
    PARTITION BY l.location_id
    ORDER BY total_orders DESC
) = 1;

In [0]:
SELECT 
    l.area,
    COUNT(*) AS cancelled_orders
FROM orders_bronze o   
JOIN locations_filtered l 
ON o.delivery_location_id=l.location_id
WHERE LOWER(order_status)='cancelled'
GROUP BY l.area
ORDER BY cancelled_orders DESC;

In [0]:
SELECT
    c.customer_id,
    c.name AS customer_name,
    COUNT(DISTINCT o.restaurant_id) AS restaurants
FROM orders_bronze o  
JOIN customers_filtered c
USING (customer_id)
GROUP BY c.customer_id, customer_name
ORDER BY restaurants DESC
LIMIT 10;

In [0]:
SELECT 
    r.restaurant_id,
    r.restaurant_name,
    o.customer_id,
    COUNT(*) AS total_orders
FROM orders_bronze o  
JOIN restaurants_filtered r 
USING(restaurant_id)
GROUP BY r.restaurant_id,r.restaurant_name,o.customer_id
HAVING COUNT(*) > 1
ORDER BY total_orders DESC;

In [0]:
SELECT 
    r.cuisines,
    o.payment_method,
    COUNT(*) AS total_orders
FROM orders_bronze o  
JOIN restaurants_filtered r 
USING(restaurant_id)
WHERE o.total_amount > 40 
GROUP BY r.cuisines,o.payment_method
QUALIFY RANK() OVER (
    PARTITION BY r.cuisines
    ORDER BY total_orders DESC
) = 1;